In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import kagglehub

# Download latest version
path = kagglehub.dataset_download("uciml/mushroom-classification")

print("Path to dataset files:", path)

100%|██████████| 34.2k/34.2k [00:00<00:00, 722kB/s]

Extracting files...
Path to dataset files: /Users/mohammadharis/.cache/kagglehub/datasets/uciml/mushroom-classification/versions/1


In [4]:
spark = (SparkSession.builder
         .appName("MushroomWorkshop")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")   # small dataset, so few partitions = faster
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")                 # hide the noisy warnings

DATA_PATH = "/Users/mohammadharis/.cache/kagglehub/datasets/uciml/mushroom-classification/versions/1"

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/27 10:54:04 WARN Utils: Your hostname, Mohammads-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.20.33 instead (on interface en0)
26/10/27 10:54:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/mohammadharis/Documents/PySpark/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/27 10:54:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
raw_df = (spark.read
          .option("header", True)
          .option("inferSchema", False)
          .csv(DATA_PATH))

df = raw_df.toDF(*[c.replace("-", "_") for c in raw_df.columns])

print(f"Rows: {df.count():,}  |  Columns: {len(df.columns)}")
df.printSchema()
df.show(5)

Rows: 8,124  |  Columns: 23
root
 |-- class: string (nullable = true)
 |-- cap_shape: string (nullable = true)
 |-- cap_surface: string (nullable = true)
 |-- cap_color: string (nullable = true)
 |-- bruises: string (nullable = true)
 |-- odor: string (nullable = true)
 |-- gill_attachment: string (nullable = true)
 |-- gill_spacing: string (nullable = true)
 |-- gill_size: string (nullable = true)
 |-- gill_color: string (nullable = true)
 |-- stalk_shape: string (nullable = true)
 |-- stalk_root: string (nullable = true)
 |-- stalk_surface_above_ring: string (nullable = true)
 |-- stalk_surface_below_ring: string (nullable = true)
 |-- stalk_color_above_ring: string (nullable = true)
 |-- stalk_color_below_ring: string (nullable = true)
 |-- veil_type: string (nullable = true)
 |-- veil_color: string (nullable = true)
 |-- ring_number: string (nullable = true)
 |-- ring_type: string (nullable = true)
 |-- spore_print_color: string (nullable = true)
 |-- population: string (nullable =

# Part 2: Exploration
2.1 Target distribution

Explanation: First check whether the classes are balanced. If one class were very rare, accuracy would be a misleading metric.

In [6]:
total = df.count()
(df.groupBy("class")
   .count()
   .withColumn("percent", F.round(F.col("count") * 100 / total, 2))
   .orderBy("class")
   .show())

+-----+-----+-------+
|class|count|percent|
+-----+-----+-------+
|    e| 4208|   51.8|
|    p| 3916|   48.2|
+-----+-----+-------+



# 2.2 Distinct values per column

Explanation: This shows how many categories each feature has and finds constant columns (only one value), which are useless for prediction. A single agg computes everything in one Spark job.

In [7]:
distinct_row = df.agg(*[F.countDistinct(c).alias(c) for c in df.columns]).first().asDict()

(spark.createDataFrame(list(distinct_row.items()), "column string, n_distinct long")
      .orderBy("n_distinct")
      .show(30))

+--------------------+----------+
|              column|n_distinct|
+--------------------+----------+
|           veil_type|         1|
|           gill_size|         2|
|         stalk_shape|         2|
|     gill_attachment|         2|
|        gill_spacing|         2|
|               class|         2|
|             bruises|         2|
|         ring_number|         3|
|stalk_surface_abo...|         4|
|stalk_surface_bel...|         4|
|          veil_color|         4|
|         cap_surface|         4|
|          stalk_root|         5|
|           ring_type|         5|
|          population|         6|
|           cap_shape|         6|
|             habitat|         7|
|   spore_print_color|         9|
|stalk_color_above...|         9|
|stalk_color_below...|         9|
|                odor|         9|
|           cap_color|        10|
|          gill_color|        12|
+--------------------+----------+



# 2.3 Missing values

Explanation: There are no real NULLs. Missing values are written as ?, so we count both.

In [8]:
missing_row = df.agg(*[
    F.sum(F.when(F.col(c).isNull() | (F.col(c) == "?"), 1).otherwise(0)).alias(c)
    for c in df.columns
]).first().asDict()

spark.createDataFrame(
    [(c, int(n), round(n * 100 / total, 2)) for c, n in missing_row.items() if n > 0],
    "column string, n_missing int, percent double").show()

+----------+---------+-------+
|    column|n_missing|percent|
+----------+---------+-------+
|stalk_root|     2480|  30.53|
+----------+---------+-------+



# 2.4 Duplicates

In [9]:
print("Duplicate rows:", total - df.dropDuplicates().count())

Duplicate rows: 0


# 2.5 Odor vs. class

Explanation: crosstab builds a table with one row per odor and one column per class.

In [10]:
df.crosstab("odor", "class").orderBy("odor_class").show()

+----------+----+----+
|odor_class|   e|   p|
+----------+----+----+
|         a| 400|   0|
|         c|   0| 192|
|         f|   0|2160|
|         l| 400|   0|
|         m|   0|  36|
|         n|3408| 120|
|         p|   0| 256|
|         s|   0| 576|
|         y|   0| 576|
+----------+----+----+



# Part 3: Cleaning

Explanation:

1. Drop constant columns (veil_type).
2. Keep ? in stalk_root as its own category. Deleting those rows would lose about 30% of the data, and since everything is categorical, “missing” can simply be one more category.

.cache() keeps the table in memory because we’ll reuse it many times.

In [11]:
constant_cols = [c for c, n in distinct_row.items() if n == 1]
print("Dropped:", constant_cols)

clean_df = df.drop(*constant_cols).cache()
print("Columns left:", len(clean_df.columns))

Dropped: ['veil_type']
Columns left: 22


# Part 4: Joins
4.0 Lookup table

Explanation: Codes like 'f' aren’t readable, so we build a table (attribute, code, label) from the dataset’s documentation. We’ll use it to practise every type of join.

In [12]:
codes = {
    "class":            {"e": "edible", "p": "poisonous"},
    "cap_shape":        {"b": "bell", "c": "conical", "x": "convex", "f": "flat", "k": "knobbed", "s": "sunken"},
    "cap_surface":      {"f": "fibrous", "g": "grooves", "y": "scaly", "s": "smooth"},
    "cap_color":        {"n": "brown", "b": "buff", "c": "cinnamon", "g": "gray", "r": "green", "p": "pink",
                         "u": "purple", "e": "red", "w": "white", "y": "yellow"},
    "bruises":          {"t": "bruises", "f": "no"},
    "odor":             {"a": "almond", "l": "anise", "c": "creosote", "y": "fishy", "f": "foul", "m": "musty",
                         "n": "none", "p": "pungent", "s": "spicy"},
    "gill_attachment":  {"a": "attached", "d": "descending", "f": "free", "n": "notched"},
    "gill_spacing":     {"c": "close", "w": "crowded", "d": "distant"},
    "gill_size":        {"b": "broad", "n": "narrow"},
    "gill_color":       {"k": "black", "n": "brown", "b": "buff", "h": "chocolate", "g": "gray", "r": "green",
                         "o": "orange", "p": "pink", "u": "purple", "e": "red", "w": "white", "y": "yellow"},
    "stalk_shape":      {"e": "enlarging", "t": "tapering"},
    "stalk_root":       {"b": "bulbous", "c": "club", "u": "cup", "e": "equal", "z": "rhizomorphs", "r": "rooted",
                         "?": "missing"},
    "veil_type":        {"p": "partial", "u": "universal"},
    "veil_color":       {"n": "brown", "o": "orange", "w": "white", "y": "yellow"},
    "ring_number":      {"n": "none", "o": "one", "t": "two"},
    "ring_type":        {"c": "cobwebby", "e": "evanescent", "f": "flaring", "l": "large", "n": "none",
                         "p": "pendant", "s": "sheathing", "z": "zone"},
    "spore_print_color":{"k": "black", "n": "brown", "b": "buff", "h": "chocolate", "r": "green", "o": "orange",
                         "u": "purple", "w": "white", "y": "yellow"},
    "population":       {"a": "abundant", "c": "clustered", "n": "numerous", "s": "scattered", "v": "several",
                         "y": "solitary"},
    "habitat":          {"g": "grasses", "l": "leaves", "m": "meadows", "p": "paths", "u": "urban", "w": "waste",
                         "d": "woods"},
}
surface = {"f": "fibrous", "y": "scaly", "k": "silky", "s": "smooth"}
stalk_color = {"n": "brown", "b": "buff", "c": "cinnamon", "g": "gray", "o": "orange", "p": "pink",
               "e": "red", "w": "white", "y": "yellow"}
codes["stalk_surface_above_ring"] = surface
codes["stalk_surface_below_ring"] = surface
codes["stalk_color_above_ring"] = stalk_color
codes["stalk_color_below_ring"] = stalk_color

lookup_df = spark.createDataFrame(
    [(attr, code, label) for attr, m in codes.items() for code, label in m.items()],
    "attribute string, code string, label string").cache()

lookup_df.filter("attribute = 'odor'").show()

+---------+----+--------+
|attribute|code|   label|
+---------+----+--------+
|     odor|   a|  almond|
|     odor|   l|   anise|
|     odor|   c|creosote|
|     odor|   y|   fishy|
|     odor|   f|    foul|
|     odor|   m|   musty|
|     odor|   n|    none|
|     odor|   p| pungent|
|     odor|   s|   spicy|
+---------+----+--------+



# 4.1 LEFT JOIN: add the label

Explanation: clean_df is the left table because it’s written first. Every mushroom is kept, and odor_label is added when a match exists. Checking that the row count stays the same confirms the join didn’t lose or duplicate any rows.

In [13]:
odor_lookup = (lookup_df.filter("attribute = 'odor'")
                        .select(F.col("code").alias("odor"), F.col("label").alias("odor_label")))

odor_decoded = clean_df.join(odor_lookup, on="odor", how="left")
odor_decoded.select("class", "odor", "odor_label").show(5)
print("Before:", clean_df.count(), "| After LEFT join:", odor_decoded.count())

+-----+----+----------+
|class|odor|odor_label|
+-----+----+----------+
|    p|   p|   pungent|
|    e|   a|    almond|
|    e|   l|     anise|
|    p|   p|   pungent|
|    e|   n|      none|
+-----+----+----------+
only showing top 5 rows
Before: 8124 | After LEFT join: 8124


# 4.2 Decode every column

Explanation: This is the same left join, repeated for each column in a loop. F.broadcast() tells Spark the lookup is small, so it gets copied to every worker. That’s much faster than reshuffling the big table.

In [14]:
decoded_df = clean_df
for c in clean_df.columns:
    lk = (lookup_df.filter(F.col("attribute") == c)
                   .select(F.col("code").alias(c), F.col("label").alias(f"{c}_label")))
    decoded_df = decoded_df.join(F.broadcast(lk), on=c, how="left")

readable_df = decoded_df.select(*[F.col(f"{c}_label").alias(c) for c in clean_df.columns]).cache()
readable_df.show(5, truncate=False)

+---------+---------+-----------+---------+-------+-------+---------------+------------+---------+----------+-----------+----------+------------------------+------------------------+----------------------+----------------------+----------+-----------+----------+-----------------+----------+-------+
|class    |cap_shape|cap_surface|cap_color|bruises|odor   |gill_attachment|gill_spacing|gill_size|gill_color|stalk_shape|stalk_root|stalk_surface_above_ring|stalk_surface_below_ring|stalk_color_above_ring|stalk_color_below_ring|veil_color|ring_number|ring_type |spore_print_color|population|habitat|
+---------+---------+-----------+---------+-------+-------+---------------+------------+---------+----------+-----------+----------+------------------------+------------------------+----------------------+----------------------+----------+-----------+----------+-----------------+----------+-------+
|poisonous|convex   |smooth     |brown    |bruises|pungent|free           |close       |narrow   |bl

# 4.3 INNER vs. LEFT vs. RIGHT

Explanation: This small table only describes 4 of the 9 odors:

INNER keeps only the mushrooms whose odor is in the table, so rows are lost.
LEFT keeps every mushroom, with description = NULL when there’s no match.
RIGHT with the tables swapped gives the same result as LEFT.

In [15]:
smell_notes = spark.createDataFrame(
    [("a", "sweet almond smell"), ("l", "anise smell"), ("f", "foul, rotten smell"), ("n", "no smell")],
    "odor string, description string")

inner_join = clean_df.join(smell_notes, "odor", "inner")
left_join  = clean_df.join(smell_notes, "odor", "left")
right_join = smell_notes.join(clean_df, "odor", "right")

print("clean_df  :", clean_df.count())
print("INNER     :", inner_join.count(), " <- rows lost")
print("LEFT      :", left_join.count(),  " <- all kept")
print("RIGHT     :", right_join.count(), " <- same as LEFT")

left_join.groupBy("odor", "description").count().orderBy("odor").show()

clean_df  : 8124
INNER     : 6488  <- rows lost
LEFT      : 8124  <- all kept
RIGHT     : 8124  <- same as LEFT
+----+------------------+-----+
|odor|       description|count|
+----+------------------+-----+
|   a|sweet almond smell|  400|
|   c|              NULL|  192|
|   f|foul, rotten smell| 2160|
|   l|       anise smell|  400|
|   m|              NULL|   36|
|   n|          no smell| 3528|
|   p|              NULL|  256|
|   s|              NULL|  576|
|   y|              NULL|  576|
+----+------------------+-----+



# 4.4 LEFT SEMI and LEFT ANTI

Explanation:

SEMI keeps the left rows that have a match. It adds no columns and creates no duplicates.
ANTI keeps the left rows that have no match.

The question here: which odors are risky (seen on at least one poisonous mushroom), and which aren’t?

In [16]:
risky_odors = clean_df.filter(F.col("class") == "p").select("odor").distinct()

risky_mushrooms      = clean_df.join(risky_odors, "odor", "left_semi")
safe_smell_mushrooms = clean_df.join(risky_odors, "odor", "left_anti")

print("Risky odors:", sorted(r.odor for r in risky_odors.collect()))
print("With risky odor :", risky_mushrooms.count())
print("With 'safe' odor:", safe_smell_mushrooms.count())
safe_smell_mushrooms.groupBy("odor", "class").count().orderBy("odor").show()

Risky odors: ['c', 'f', 'm', 'n', 'p', 's', 'y']
With risky odor : 7324
With 'safe' odor: 800
+----+-----+-----+
|odor|class|count|
+----+-----+-----+
|   a|    e|  400|
|   l|    e|  400|
+----+-----+-----+



Explanation: ANTI is also useful for checking data quality. We look for codes in the data that aren’t in the documentation. unpivot turns the columns into (attribute, code) rows. An empty result means every code is documented.

In [17]:
data_codes = (clean_df
              .unpivot(ids=[], values=clean_df.columns, variableColumnName="attribute", valueColumnName="code")
              .distinct())

undocumented = data_codes.join(lookup_df, on=["attribute", "code"], how="left_anti")
print("Undocumented codes:", undocumented.count())
undocumented.show()

Undocumented codes: 0
+---------+----+
|attribute|code|
+---------+----+
+---------+----+



# 4.5 FULL OUTER: compare two tables

Explanation: We count edible and poisonous mushrooms per spore color in two separate tables. FULL OUTER keeps colors that appear in only one of the two tables, filling in NULL, which we then replace with 0. An INNER join would drop those colors, and they’re the most interesting ones.

In [18]:
edible_by_spore    = clean_df.filter("class = 'e'").groupBy("spore_print_color").agg(F.count("*").alias("n_edible"))
poisonous_by_spore = clean_df.filter("class = 'p'").groupBy("spore_print_color").agg(F.count("*").alias("n_poisonous"))

(edible_by_spore
   .join(poisonous_by_spore, "spore_print_color", "full_outer")
   .fillna(0, subset=["n_edible", "n_poisonous"])
   .withColumn("pct_poisonous",
               F.round(F.col("n_poisonous") * 100 / (F.col("n_edible") + F.col("n_poisonous")), 1))
   .orderBy(F.desc("pct_poisonous"))
   .show())

+-----------------+--------+-----------+-------------+
|spore_print_color|n_edible|n_poisonous|pct_poisonous|
+-----------------+--------+-----------+-------------+
|                r|       0|         72|        100.0|
|                h|      48|       1584|         97.1|
|                w|     576|       1812|         75.9|
|                k|    1648|        224|         12.0|
|                n|    1744|        224|         11.4|
|                b|      48|          0|          0.0|
|                o|      48|          0|          0.0|
|                u|      48|          0|          0.0|
|                y|      48|          0|          0.0|
+-----------------+--------+-----------+-------------+



# 4.6 CROSS JOIN: every combination

Explanation: CROSS builds the full grid of 9 odors × 7 habitats = 63 combinations. Then a LEFT join adds the real counts, and combinations that never occur get 0.

In [19]:
odors    = lookup_df.filter("attribute = 'odor'").select(F.col("code").alias("odor"))
habitats = lookup_df.filter("attribute = 'habitat'").select(F.col("code").alias("habitat"))

grid = odors.crossJoin(habitats)
print("Grid size:", grid.count())

counts = clean_df.groupBy("odor", "habitat").count()
odor_habitat = grid.join(counts, ["odor", "habitat"], "left").fillna(0, subset=["count"])

print("Combinations that never occur:", odor_habitat.filter("count = 0").count())
odor_habitat.groupBy("odor").pivot("habitat").sum("count").orderBy("odor").show()

Grid size: 63
Combinations that never occur: 33
+----+----+----+---+---+---+---+---+
|odor|   d|   g|  l|  m|  p|  u|  w|
+----+----+----+---+---+---+---+---+
|   a|  48| 176|  0|128| 48|  0|  0|
|   c| 192|   0|  0|  0|  0|  0|  0|
|   f| 624| 576|192|  0|624|144|  0|
|   l|  48| 176|  0|128| 48|  0|  0|
|   m|  36|   0|  0|  0|  0|  0|  0|
|   n|1816|1092|256| 36| 40| 96|192|
|   p|   0| 128|  0|  0|  0|128|  0|
|   s| 192|   0|192|  0|192|  0|  0|
|   y| 192|   0|192|  0|192|  0|  0|
+----+----+----+---+---+---+---+---+



# Part 5: Spark SQL

Explanation: We register the tables as SQL “views” so we can query them with regular SQL.

In [20]:
readable_df.createOrReplaceTempView("mushrooms")
lookup_df.createOrReplaceTempView("lookup")

# Q1: % poisonous by odor
spark.sql("""
    SELECT odor, COUNT(*) AS n,
           ROUND(AVG(CASE WHEN class = 'poisonous' THEN 1 ELSE 0 END) * 100, 1) AS pct_poisonous
    FROM mushrooms
    GROUP BY odor
    ORDER BY pct_poisonous DESC
""").show()

+--------+----+-------------+
|    odor|   n|pct_poisonous|
+--------+----+-------------+
|creosote| 192|        100.0|
|   spicy| 576|        100.0|
|   fishy| 576|        100.0|
|    foul|2160|        100.0|
|   musty|  36|        100.0|
| pungent| 256|        100.0|
|    none|3528|          3.4|
|  almond| 400|          0.0|
|   anise| 400|          0.0|
+--------+----+-------------+



In [21]:
# Q2: odorless mushrooms – which spore colors are dangerous?
spark.sql("""
    SELECT spore_print_color, COUNT(*) AS n,
           ROUND(AVG(CASE WHEN class = 'poisonous' THEN 1 ELSE 0 END) * 100, 1) AS pct_poisonous
    FROM mushrooms
    WHERE odor = 'none'
    GROUP BY spore_print_color
    ORDER BY pct_poisonous DESC
""").show()

+-----------------+----+-------------+
|spore_print_color|   n|pct_poisonous|
+-----------------+----+-------------+
|            green|  72|        100.0|
|            white| 624|          7.7|
|            brown|1344|          0.0|
|        chocolate|  48|          0.0|
|           yellow|  48|          0.0|
|            black|1296|          0.0|
|           orange|  48|          0.0|
|             buff|  48|          0.0|
+-----------------+----+-------------+



In [22]:
# Q3: LEFT ANTI in SQL – habitats never seen on a poisonous mushroom
spark.sql("""
    SELECT l.label AS habitat
    FROM lookup l
    LEFT ANTI JOIN (SELECT DISTINCT habitat FROM mushrooms WHERE class = 'poisonous') p
        ON l.label = p.habitat
    WHERE l.attribute = 'habitat'
""").show()

+-------+
|habitat|
+-------+
|  waste|
+-------+



In [26]:
# Q4: simple rule "poisonous unless odor is almond, anise or none" – accuracy?
spark.sql("""
    SELECT ROUND(AVG(CASE WHEN (odor NOT IN ('almond','anise','none')) = (class = 'poisonous')
                          THEN 1 ELSE 0 END) * 100, 2) AS accuracy_pct
    FROM mushrooms
""").show()
print('One rule based only on odor already gives about 98.5% accuracy.')

+------------+
|accuracy_pct|
+------------+
|       98.52|
+------------+

One rule based only on odor already gives about 98.5% accuracy.


# Part 6: Machine learning
6.1 Preparing the features

Explanation: Models need numbers, not letters. The steps are:

1. StringIndexer converts each letter into a number (e → 0, p → 1 for the label).
2. OneHotEncoder turns each number into a 0/1 vector, so the model doesn’t think “category 3 > category 1”.
3. VectorAssembler combines everything into a single features column.

A Pipeline chains these steps so the same transformations are applied to the training data and the test data.

In [27]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

feature_cols = [c for c in clean_df.columns if c != "class"]
idx_cols = [f"{c}_idx" for c in feature_cols]
ohe_cols = [f"{c}_ohe" for c in feature_cols]

label_indexer   = StringIndexer(inputCol="class", outputCol="label", stringOrderType="alphabetAsc")  # e=0, p=1
feature_indexer = StringIndexer(inputCols=feature_cols, outputCols=idx_cols, handleInvalid="keep")
encoder         = OneHotEncoder(inputCols=idx_cols, outputCols=ohe_cols, handleInvalid="keep")
assembler       = VectorAssembler(inputCols=ohe_cols, outputCol="features")

# 6.2 Train / test split

Explanation: 80% of the data trains the model. The other 20% is kept aside to test it on data it has never seen. seed=42 gives the same split every time you run it.

In [28]:
train_df, test_df = clean_df.randomSplit([0.8, 0.2], seed=42)
train_df.cache(); test_df.cache()
print("Train:", train_df.count(), "| Test:", test_df.count())

Train: 6573 | Test: 1551


# 6.3 Evaluation function

Explanation:

1. Accuracy: % of predictions that are correct.
2. AUC: 1.0 means the model separates the two classes perfectly.
3. Recall (poisonous): % of poisonous mushrooms that were detected. This is the most important metric here. Calling a poisonous mushroom “edible” (a false negative) is dangerous, while the opposite mistake only wastes a mushroom.
4. Confusion matrix: rows are the true class, columns are the predicted class.

In [29]:
auc_eval = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
mc_eval  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")

def evaluate(pred, name):
    m = {
        "model":            name,
        "accuracy":         mc_eval.evaluate(pred, {mc_eval.metricName: "accuracy"}),
        "f1":               mc_eval.evaluate(pred, {mc_eval.metricName: "f1"}),
        "auc":              auc_eval.evaluate(pred),
        "recall_poisonous": mc_eval.evaluate(pred, {mc_eval.metricName: "recallByLabel", mc_eval.metricLabel: 1.0}),
        "false_negatives":  pred.filter("label = 1 AND prediction = 0").count(),
    }
    for k, v in m.items():
        print(f"{k:>18}: {v:.4f}" if isinstance(v, float) else f"{k:>18}: {v}")
    print("\nConfusion matrix (0 = edible, 1 = poisonous)")
    pred.groupBy("label").pivot("prediction", [0.0, 1.0]).count().fillna(0).orderBy("label").show()
    return m

# 6.4 Model 1: Logistic Regression

Explanation: A simple linear model that makes a good starting point (baseline).

In [30]:
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=50)
lr_pipeline = Pipeline(stages=[label_indexer, feature_indexer, encoder, assembler, lr])

lr_model = lr_pipeline.fit(train_df)
lr_pred  = lr_model.transform(test_df)

lr_pred.select("class", "label", "prediction", "probability").show(5, truncate=False)
lr_metrics = evaluate(lr_pred, "Logistic Regression")

+-----+-----+----------+-------------------------------------------+
|class|label|prediction|probability                                |
+-----+-----+----------+-------------------------------------------+
|e    |0.0  |0.0       |[0.9999999999969165,3.0835334285939098E-12]|
|e    |0.0  |0.0       |[0.9999999999999627,3.730349362740526E-14] |
|e    |0.0  |0.0       |[0.9999999999863236,1.3676393351147453E-11]|
|e    |0.0  |0.0       |[0.9999999999993996,6.004086117172847E-13] |
|e    |0.0  |0.0       |[0.9999999999858946,1.4105383527862614E-11]|
+-----+-----+----------+-------------------------------------------+
only showing top 5 rows
             model: Logistic Regression
          accuracy: 1.0000
                f1: 1.0000
               auc: 1.0000
  recall_poisonous: 1.0000
   false_negatives: 0

Confusion matrix (0 = edible, 1 = poisonous)
+-----+---+---+
|label|0.0|1.0|
+-----+---+---+
|  0.0|788|  0|
|  1.0|  0|763|
+-----+---+---+



# 6.5 Model 2: Random Forest

Explanation: A random forest is a group of decision trees that vote. Trees can work directly with categories, so one-hot encoding isn’t needed. We give it the indexed columns, which also lets us see how important each original feature is.

In [31]:
rf_assembler = VectorAssembler(inputCols=idx_cols, outputCol="features")
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50, maxDepth=8, seed=42)
rf_pipeline = Pipeline(stages=[label_indexer, feature_indexer, rf_assembler, rf])

rf_model = rf_pipeline.fit(train_df)
rf_pred  = rf_model.transform(test_df)
rf_metrics = evaluate(rf_pred, "Random Forest")

             model: Random Forest
          accuracy: 1.0000
                f1: 1.0000
               auc: 1.0000
  recall_poisonous: 1.0000
   false_negatives: 0

Confusion matrix (0 = edible, 1 = poisonous)
+-----+---+---+
|label|0.0|1.0|
+-----+---+---+
|  0.0|788|  0|
|  1.0|  0|763|
+-----+---+---+



# 6.6 Feature importance

In [32]:
importances = rf_model.stages[-1].featureImportances.toArray()
(spark.createDataFrame([(c, float(v)) for c, v in zip(feature_cols, importances)],
                       "feature string, importance double")
      .orderBy(F.desc("importance"))
      .show(25))

+--------------------+--------------------+
|             feature|          importance|
+--------------------+--------------------+
|                odor|   0.475203731603646|
|   spore_print_color|  0.0956326955684917|
|stalk_surface_abo...| 0.07183423054576439|
|          gill_color|  0.0646960330710458|
|stalk_surface_bel...| 0.05216564199735419|
|           gill_size| 0.05136906920875701|
|           ring_type| 0.04029904118724666|
|          population| 0.03915263235770935|
|          stalk_root|0.019325444768846105|
|         ring_number|0.013470537799514307|
|           cap_color|0.013186647174862403|
|             habitat|0.011265198815533884|
|             bruises|0.010800998196595404|
|stalk_color_below...|0.010498971854755049|
|        gill_spacing|0.009106047693571605|
|         stalk_shape|0.007606307197743435|
|stalk_color_above...|0.006093368383117247|
|         cap_surface| 0.00417802727529438|
|           cap_shape|0.003154357351853672|
|          veil_color|5.79194238

# 6.7 Without odor and spore color

Explanation: We remove the two strongest features to see whether the other physical features are still enough to predict the class.

In [33]:
reduced_cols = [c for c in feature_cols if c not in ("odor", "spore_print_color")]
reduced_idx  = [f"{c}_idx" for c in reduced_cols]

reduced_pipeline = Pipeline(stages=[
    label_indexer,
    StringIndexer(inputCols=reduced_cols, outputCols=reduced_idx, handleInvalid="keep"),
    VectorAssembler(inputCols=reduced_idx, outputCol="features"),
    RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50, maxDepth=8, seed=42),
])
reduced_pred = reduced_pipeline.fit(train_df).transform(test_df)
reduced_metrics = evaluate(reduced_pred, "RF without odor/spore")

             model: RF without odor/spore
          accuracy: 1.0000
                f1: 1.0000
               auc: 1.0000
  recall_poisonous: 1.0000
   false_negatives: 0

Confusion matrix (0 = edible, 1 = poisonous)
+-----+---+---+
|label|0.0|1.0|
+-----+---+---+
|  0.0|788|  0|
|  1.0|  0|763|
+-----+---+---+



# 6.8 (Optional) Tuning with cross-validation

Explanation: This tests several parameter combinations. For each one, the training data is split into 3 folds: the model trains on 2 and is checked on the third. The combination with the best AUC is kept.

In [34]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

grid = (ParamGridBuilder()
        .addGrid(lr.regParam, [0.0, 0.01, 0.1])
        .addGrid(lr.elasticNetParam, [0.0, 1.0])
        .build())

cv = CrossValidator(estimator=lr_pipeline, estimatorParamMaps=grid,
                    evaluator=auc_eval, numFolds=3, seed=42, parallelism=2)
cv_model = cv.fit(train_df)

best = cv_model.bestModel.stages[-1]
print("Best regParam:", best.getRegParam(), "| elasticNetParam:", best.getElasticNetParam())
cv_metrics = evaluate(cv_model.transform(test_df), "LR tuned")

Best regParam: 0.01 | elasticNetParam: 0.0
             model: LR tuned
          accuracy: 0.9994
                f1: 0.9994
               auc: 1.0000
  recall_poisonous: 0.9987
   false_negatives: 1

Confusion matrix (0 = edible, 1 = poisonous)
+-----+---+---+
|label|0.0|1.0|
+-----+---+---+
|  0.0|788|  0|
|  1.0|  1|762|
+-----+---+---+



# 6.9 Compare the models

In [35]:
spark.createDataFrame(
    [(m["model"], round(m["accuracy"], 4), round(m["auc"], 4), round(m["recall_poisonous"], 4), m["false_negatives"])
     for m in [lr_metrics, rf_metrics, reduced_metrics, cv_metrics]],
    "model string, accuracy double, auc double, recall_poisonous double, false_negatives long").show(truncate=False)

+---------------------+--------+---+----------------+---------------+
|model                |accuracy|auc|recall_poisonous|false_negatives|
+---------------------+--------+---+----------------+---------------+
|Logistic Regression  |1.0     |1.0|1.0             |0              |
|Random Forest        |1.0     |1.0|1.0             |0              |
|RF without odor/spore|1.0     |1.0|1.0             |0              |
|LR tuned             |0.9994  |1.0|0.9987          |1              |
+---------------------+--------+---+----------------+---------------+



# Two checks that prove their is no overfitting

1. Compare train vs. test accuracy. If they’re almost the same, there’s no overfitting.

In [36]:
for name, model in [("LR", lr_model), ("RF", rf_model)]:
    train_acc = mc_eval.evaluate(model.transform(train_df), {mc_eval.metricName: "accuracy"})
    test_acc  = mc_eval.evaluate(model.transform(test_df),  {mc_eval.metricName: "accuracy"})
    print(f"{name}: train = {train_acc:.4f} | test = {test_acc:.4f}")

LR: train = 1.0000 | test = 1.0000
RF: train = 1.0000 | test = 1.0000


2. Try several different splits. If you get 100% every time, the result isn’t luck from one particular split.

In [37]:
for seed in [1, 7, 42, 123, 2024]:
    tr, te = clean_df.randomSplit([0.8, 0.2], seed=seed)
    pred = rf_pipeline.fit(tr).transform(te)
    acc = mc_eval.evaluate(pred, {mc_eval.metricName: "accuracy"})
    print(f"seed {seed}: test accuracy = {acc:.4f}")

seed 1: test accuracy = 1.0000
seed 7: test accuracy = 1.0000
seed 42: test accuracy = 1.0000
seed 123: test accuracy = 1.0000
seed 2024: test accuracy = 1.0000
